In [0]:
%sql
-- Semantic layer, step 1: regular SQL view — one row per sales order line (no aggregation).
-- Joins sales_order_detail to product to map ProductLine to a readable category and derive the order quarter.
CREATE OR REPLACE VIEW adventure_works.gold.sales_lines_vw AS
SELECT
  CAST(d.SalesOrderID AS INT)                      AS SalesOrderID,
  CASE
    WHEN TRIM(p.ProductLine) = 'R' THEN 'Road'
    WHEN TRIM(p.ProductLine) = 'M' THEN 'Mountain'
    WHEN TRIM(p.ProductLine) = 'S' THEN 'Standard'
    WHEN TRIM(p.ProductLine) = 'T' THEN 'Touring'
    ELSE 'Other'
  END                                              AS category,
  CONCAT(
    YEAR(CAST(d.ModifiedDate AS TIMESTAMP)),
    '-Q',
    QUARTER(CAST(d.ModifiedDate AS TIMESTAMP))
  )                                               AS quarter,
  CAST(d.LineTotal AS DECIMAL(19,4))               AS LineTotal,
  CAST(d.OrderQty AS INT)                         AS OrderQty
FROM adventure_works.silver.sales_order_detail AS d
JOIN adventure_works.silver.product AS p
  ON d.ProductID = p.ProductID

In [0]:
%sql
-- Semantic layer, step 2: Unity Catalog metric view with centrally-defined, reusable KPIs.
-- Same category mapping and quarter derivation as sales_lines_vw, but measures are defined once in YAML.
CREATE OR REPLACE VIEW adventure_works.gold.sales_metrics
WITH METRICS
LANGUAGE YAML
AS $$
  version: 1.1
  source: adventure_works.silver.sales_order_detail
  joins:
    - name: product
      source: adventure_works.silver.product
      on: source.ProductID = product.ProductID
  dimensions:
    - name: category
      expr: |-
        CASE
          WHEN TRIM(product.ProductLine) = 'R' THEN 'Road'
          WHEN TRIM(product.ProductLine) = 'M' THEN 'Mountain'
          WHEN TRIM(product.ProductLine) = 'S' THEN 'Standard'
          WHEN TRIM(product.ProductLine) = 'T' THEN 'Touring'
          ELSE 'Other'
        END
    - name: quarter
      expr: CONCAT(YEAR(CAST(ModifiedDate AS TIMESTAMP)), '-Q', QUARTER(CAST(ModifiedDate AS TIMESTAMP)))
  measures:
    - name: Total Revenue
      expr: SUM(CAST(LineTotal AS DECIMAL(19,4)))
    - name: Order Count
      expr: COUNT(DISTINCT SalesOrderID)
    - name: Avg Order Value
      expr: SUM(CAST(LineTotal AS DECIMAL(19,4))) / COUNT(DISTINCT SalesOrderID)
  comment: Sales KPIs (revenue, order count, average order value) by product category and quarter
$$

In [0]:
%sql
-- Comparison 1/4: line-level view — total revenue as SUM(LineTotal) by category.
SELECT category, SUM(LineTotal) FROM adventure_works.gold.sales_lines_vw GROUP BY category;

In [0]:
%sql
-- Comparison 2/4: metric view — same revenue via MEASURE(`Total Revenue`) by category.
SELECT category, MEASURE(`Total Revenue`) FROM adventure_works.gold.sales_metrics GROUP BY category;

In [0]:
%sql
-- Comparison 3/4: line-level view — AVG(LineTotal) by category (misleading: averages order LINES).
SELECT category, AVG(LineTotal) FROM adventure_works.gold.sales_lines_vw GROUP BY category;

In [0]:
%sql
-- Comparison 4/4: metric view — MEASURE(`Avg Order Value`) by category (true average per order).
SELECT category, MEASURE(`Avg Order Value`) FROM adventure_works.gold.sales_metrics GROUP BY category;

**Why we trust the Metric View for `Avg Order Value`:** `AVG(LineTotal)` on the line-level view averages individual order *lines* (a multi-line order is counted once per line), whereas the Metric View divides total revenue by the number of distinct orders — `SUM(LineTotal) / COUNT(DISTINCT SalesOrderID)` — giving the true average order value.

**Why we trust the Metric View for `Avg Order Value`:** `AVG(LineTotal)` on the line-level view averages individual order *lines* (a multi-line order is counted once per line), whereas the Metric View divides total revenue by the number of distinct orders — `SUM(LineTotal) / COUNT(DISTINCT SalesOrderID)` — giving the true average order value.